# Amazon Review Sentiment Analysis — Model Training

## What this notebook does
Trains a 3-class sentiment classifier (positive / neutral / negative) on Amazon food reviews.

## Approach
- **Features:** TF-IDF on lemmatized text enriched with bigrams and trigrams
- **Bias fix:** Class weights for LR, undersampling for SVC and NaiveBayes
- **Models compared:** Logistic Regression, OneVsRest LinearSVC, Naive Bayes
- **Selection metric:** Weighted F1 on validation set
- **Final metric:** Weighted F1 + balanced accuracy on held-out test set

## Why TF-IDF over GloVe here
GloVe with average pooling loses word order and sentiment direction (e.g. 'not good' averages the same as 'good').
TF-IDF with bigrams and trigrams explicitly captures multi-word patterns and consistently
outperforms average-pooled embeddings on short opinionated text like product reviews.

## Cell 1 — Environment Setup and Spark Session

We point the JVM to Java 17 before any Spark import.
Java 17 requires explicit module-access flags for Spark's internal reflection (the --add-opens lines).
Without them Spark crashes silently on Apple Silicon.

- `driver.memory = 6g` — safe for 8GB free RAM, leaves headroom for OS and Python
- `setLogLevel(ERROR)` — suppresses noisy WARN/INFO so notebook output stays readable

In [1]:
import os

# Must be set BEFORE any PySpark import — JVM reads this at startup
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"]      = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ.get("PATH", "")

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import (
    LogisticRegression, LinearSVC, OneVsRest, NaiveBayes
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import json

# Java 17 module-access flags required by Spark internal reflection
_jvm_opts = " ".join([
    "--add-opens=java.base/java.lang=ALL-UNNAMED",
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED",
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED",
    "--add-opens=java.base/java.io=ALL-UNNAMED",
    "--add-opens=java.base/java.net=ALL-UNNAMED",
    "--add-opens=java.base/java.nio=ALL-UNNAMED",
    "--add-opens=java.base/java.util=ALL-UNNAMED",
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED",
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED",
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED",
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED",
])

spark = (
    SparkSession.builder
    .appName("AmazonSentiment_TFIDF")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .config("spark.driver.extraJavaOptions", _jvm_opts)
    .config("spark.executor.extraJavaOptions", _jvm_opts)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version : {spark.version}")
print(f"Java home     : {os.environ['JAVA_HOME']}")

26/05/08 00:13:29 WARN Utils: Your hostname, Louays-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.25 instead (on interface en0)
26/05/08 00:13:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 00:14:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.1
Java home     : /opt/homebrew/opt/openjdk@17


## Cell 2 — Load Preprocessed Datasets

Loads the three CSV files produced by 01_eda_preprocessing.ipynb.

Columns kept:
- `clean_text`  — lemmatized + bigram + trigram enriched text (model input)
- `sentiment`   — target label: negative / neutral / positive
- `ProductId`, `UserId`, `Id`, `Time` — needed later for MongoDB and dashboard queries

`repartition(8)` spreads data across 8 parallel tasks — speeds up all downstream operations.

In [2]:
BASE = "/Users/beethoven/BigData/AmazonReview"
COLS = ["clean_text", "sentiment", "ProductId", "UserId", "Id", "Time"]

def load(path):
    # spark.read.csv avoids the pandas→JVM socket transfer that causes
    # "No buffer space available" on macOS local mode.
    #
    # escape='"' is required: pandas writes RFC 4180 double-quote escaping
    # (e.g. ProfileName = "CG ""CG"""). Spark's default escape is backslash,
    # so without this option it misparses quoted fields and shifts all columns,
    # causing 'sentiment' to receive raw review text instead of the label.
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(path)
        .select(COLS)
        .dropna()
    )

train = load(f"{BASE}/data/train.csv")
val   = load(f"{BASE}/data/val.csv")
test  = load(f"{BASE}/data/test.csv")

print(f"Train : {train.count():,}")
print(f"Val   : {val.count():,}")
print(f"Test  : {test.count():,}")

print("\nClass distribution in training set:")
train.groupBy("sentiment").count().orderBy("sentiment").show()

Train : 314,900
Val   : 39,319
Test  : 39,358

Class distribution in training set:
+---------+------+
|sentiment| count|
+---------+------+
| negative| 45659|
|  neutral| 23806|
| positive|245435|
+---------+------+



## Cell 3 — Handle Class Imbalance

The dataset is heavily skewed: ~78% positive, ~14% negative, ~8% neutral.
Without correction models learn to always predict positive and still score 78% accuracy — useless.

### Strategy 1 — Class Weights (Logistic Regression)
Mathematically penalises the model more for misclassifying rare classes.
Formula: weight = total_samples / (num_classes x class_count)
Result: neutral gets ~4.4x more penalty weight than positive.

### Strategy 2 — Undersampling (SVC and Naive Bayes)
Caps every class at the size of the smallest class (neutral ~23k rows).
Used because LinearSVC and NaiveBayes do not support weightCol in Spark MLlib.

In [3]:
# Strategy 1: compute class weights
label_counts = train.groupBy("sentiment").count().collect()
total_count  = sum(r["count"] for r in label_counts)
num_classes  = len(label_counts)

weight_map = {
    r["sentiment"]: total_count / (num_classes * r["count"])
    for r in label_counts
}

print("Class weights (higher = penalised more for errors on this class):")
for sentiment, w in sorted(weight_map.items()):
    print(f"  {sentiment:10s} -> {w:.4f}")

# Join weights onto training DataFrame as a new column
weights_df = spark.createDataFrame(
    [(k, float(v)) for k, v in weight_map.items()],
    ["sentiment", "classWeight"]
)
train_w = train.join(weights_df, on="sentiment", how="left")

# Strategy 2: balanced undersampling
min_count = min(r["count"] for r in label_counts)
balanced_train = (
    train.filter(F.col("sentiment") == "positive").limit(min_count)
    .union(train.filter(F.col("sentiment") == "negative").limit(min_count))
    .union(train.filter(F.col("sentiment") == "neutral" ).limit(min_count))
)

print(f"\nBalanced sample : {min_count:,} rows per class")
print(f"Total balanced  : {balanced_train.count():,}")

Class weights (higher = penalised more for errors on this class):
  negative   -> 2.2989
  neutral    -> 4.4093
  positive   -> 0.4277

Balanced sample : 23,806 rows per class


Total balanced  : 71,418


## Cell 4 — Build TF-IDF Feature Pipeline

Converts raw text into numerical features that ML classifiers can use.

Pipeline stages:
| Stage | What it does |
|---|---|
| Tokenizer | Splits clean_text into individual word tokens |
| StopWordsRemover | Removes common words — negations already preserved in preprocessing |
| HashingTF | Maps tokens to a 200,000-dim sparse vector using hashing trick |
| IDF | Downweights tokens appearing in many documents (common = less informative) |
| StringIndexer | Converts sentiment string to numeric label 0/1/2 |

Key parameter choices:
- numFeatures=200,000 — large enough for unigrams + bigrams + trigrams without hash collisions
- minDocFreq=3 — drops tokens appearing in fewer than 3 documents (noise removal)

In [4]:
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")

remover = StopWordsRemover(inputCol="words", outputCol="filtered")

hashingTF = HashingTF(
    inputCol="filtered",
    outputCol="rawFeatures",
    numFeatures=80000
)

idf = IDF(inputCol="rawFeatures", outputCol="features", minDocFreq=3)

# handleInvalid default ("error") is correct — val/test have the same 3 labels as train.
# "keep" was wrong: it adds a 4th phantom label bucket which broke evaluation.
indexer = StringIndexer(inputCol="sentiment", outputCol="label")

FEATURE_STAGES = [tokenizer, remover, hashingTF, idf, indexer]
print("Feature pipeline: Tokenizer -> StopWordsRemover -> HashingTF(80k) -> IDF -> StringIndexer")
print("Min doc freq : 3")

Feature pipeline: Tokenizer -> StopWordsRemover -> HashingTF(80k) -> IDF -> StringIndexer
Min doc freq : 3


## Cell 5 — Train All Models

We train 3 classifiers and compare on validation set using weighted F1.
Weighted F1 accounts for class imbalance — accuracy alone is misleading here.

### Model 1 — Logistic Regression (weighted)
- Trained on full dataset with class weights
- family=multinomial: native 3-class support, no wrapper needed
- maxIter=300: enough iterations to converge on 200k features
- regParam=0.001: light regularisation, richer features tolerate this
- elasticNetParam=0.0: pure L2 (ridge), better for dense TF-IDF vectors

### Model 2 — OneVsRest LinearSVC (balanced)
- LinearSVC is binary-only so OneVsRest trains 3 separate binary classifiers
- Trained on balanced undersampled data since LinearSVC has no weightCol
- LinearSVC often outperforms LR on high-dimensional sparse text features

### Model 3 — Naive Bayes (balanced)
- Classic baseline for text classification, fast and interpretable
- smoothing=1.0: Laplace smoothing prevents zero probabilities for unseen tokens
- Trained on balanced data since NaiveBayes has no weightCol

Expected runtime: 20 to 35 minutes total.

In [5]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

results        = {}
trained_models = {}

def train_and_eval(name, classifier, train_df):
    """Build full pipeline, fit on train_df, evaluate on val set, store result."""
    print(f"\n{'='*55}")
    print(f" Training: {name}")
    print(f"{'='*55}")
    try:
        pipeline = Pipeline(stages=FEATURE_STAGES + [classifier])
        model    = pipeline.fit(train_df)
        f1       = evaluator.evaluate(model.transform(val))
        results[name]        = f1
        trained_models[name] = model
        print(f" Val F1: {f1:.4f}")
    except Exception as exc:
        print(f" Failed: {exc}")

# Model 1: Logistic Regression with class weights
lr = LogisticRegression(
    maxIter=200,             # reduced from 300 — enough for 80k features
    regParam=0.01,           # slightly more regularisation to match smaller feature space
    elasticNetParam=0.0,
    family="multinomial",
    weightCol="classWeight"
)
train_and_eval("LogisticRegression", lr, train_w)

# Model 2: OneVsRest LinearSVC on balanced data
svc = LinearSVC(maxIter=200, regParam=0.001)
ovr = OneVsRest(classifier=svc)
train_and_eval("OneVsRest_LinearSVC", ovr, balanced_train)

# Model 3: Naive Bayes on balanced data
nb = NaiveBayes(smoothing=1.0, modelType="multinomial")
train_and_eval("NaiveBayes", nb, balanced_train)

print("\n" + "="*55)
print(" RESULTS SUMMARY (Validation Weighted F1)")
print("="*55)
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:30s} : {score:.4f}")


 Training: LogisticRegression


 Val F1: 0.8351

 Training: OneVsRest_LinearSVC


 Val F1: 0.7484

 Training: NaiveBayes


 Val F1: 0.7793

 RESULTS SUMMARY (Validation Weighted F1)
  LogisticRegression             : 0.8351
  NaiveBayes                     : 0.7793
  OneVsRest_LinearSVC            : 0.7484


In [6]:
# Tune LR specifically to boost neutral recall
# Lower regParam gives the model more freedom to learn neutral patterns
lr_tuned = LogisticRegression(
    maxIter=300,
    regParam=0.003,          # between 0.01 and 0.001 — sweet spot
    elasticNetParam=0.1,     # slight L1 mix — helps sparse features like neutral
    family="multinomial",
    weightCol="classWeight"
)
train_and_eval("LogisticRegression_Tuned", lr_tuned, train_w)


 Training: LogisticRegression_Tuned


 Val F1: 0.8492


## Cell 6 — Select Best Model and Full Test Evaluation

Best model selected by highest validation F1.
Full diagnostic run on the held-out test set — data never seen during training or model selection.

Metrics reported:
- Weighted F1: primary metric, balances precision and recall accounting for class sizes
- Accuracy: raw correct predictions, reported for completeness but misleading with imbalance
- Weighted Precision / Recall: quality breakdown per class
- Per-class recall: most revealing — shows which sentiment the model struggles with
- Balanced accuracy (macro recall): average recall across all 3 classes equally weighted,
  the most honest single metric for imbalanced multi-class problems

In [7]:
best_name  = max(results, key=results.get)
best_model = trained_models[best_name]

print(f"Best model : {best_name}")
print(f"Val F1     : {results[best_name]:.4f}")
print()

test_preds = best_model.transform(test)

print("Test Set Metrics:")
print("-" * 40)
for metric in ["f1", "accuracy", "weightedPrecision", "weightedRecall"]:
    evaluator.setMetricName(metric)
    print(f"  {metric:22s} : {evaluator.evaluate(test_preds):.4f}")

per_class = (
    test_preds
    .groupBy("label")
    .agg(
        F.sum(F.when(F.col("label") == F.col("prediction"), 1).otherwise(0)).alias("tp"),
        F.count("*").alias("support")
    )
    .withColumn("recall", F.col("tp") / F.col("support"))
    .orderBy("label")
)

print("\nPer-class Recall:")
per_class.select("label", "support", "recall").show(truncate=False)

balanced_acc = per_class.select(F.avg("recall")).first()[0]
print(f"Balanced Accuracy (macro recall) : {balanced_acc:.4f}")
print()
print("Target: >0.65 is good, >0.70 is very good for 3-class imbalanced sentiment.")

Best model : LogisticRegression_Tuned
Val F1     : 0.8492

Test Set Metrics:
----------------------------------------


  f1                     : 0.8451


  accuracy               : 0.8337


  weightedPrecision      : 0.8605


  weightedRecall         : 0.8337

Per-class Recall:


+-----+-------+------------------+
|label|support|recall            |
+-----+-------+------------------+
|0.0  |30676  |0.8903377233016039|
|1.0  |5707   |0.7375153320483616|
|2.0  |2975   |0.4346218487394958|
+-----+-------+------------------+



Balanced Accuracy (macro recall) : 0.6875

Target: >0.65 is good, >0.70 is very good for 3-class imbalanced sentiment.


## Cell 7 — Save Best Model

Saves the complete trained pipeline using Spark native format.
This includes all feature stages and the classifier in one serialized object.
The streaming consumer loads this exact model to make real-time predictions.

write().overwrite().save() is safe to re-run — overwrites any previous save.

In [8]:
MODEL_PATH = f"{BASE}/model/best_sentiment_model"

best_model.write().overwrite().save(MODEL_PATH)

meta = {
    "best_model": best_name,
    "val_f1": results[best_name],
    "model_path": MODEL_PATH
}
with open(f"{BASE}/model/model_info.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Model saved  : {MODEL_PATH}")
print(f"Model type   : {best_name}")
print(f"Val F1       : {results[best_name]:.4f}")

Model saved  : /Users/beethoven/BigData/AmazonReview/model/best_sentiment_model
Model type   : LogisticRegression_Tuned
Val F1       : 0.8492


## Cell 8 — Save Label Mapping

Spark ML classifiers output numeric predictions: 0.0, 1.0, 2.0.
We need to know which number maps to which sentiment word.

StringIndexer sorts labels by frequency so label 0 = most frequent class (positive).
This JSON file is used by:
- The Kafka Spark Streaming consumer to convert predictions back to readable labels
- The Flask dashboard to display sentiment correctly

StringIndexer is always at pipeline stage index 4:
tokenizer(0), remover(1), hashingTF(2), idf(3), indexer(4)

In [9]:
# StringIndexer is always at index 4 in our FEATURE_STAGES list
indexer_model = best_model.stages[4]
labels        = indexer_model.labels
label_mapping = {str(i): label for i, label in enumerate(labels)}

print("Label mapping (numeric prediction -> sentiment string):")
for idx, label in label_mapping.items():
    count = next(r["count"] for r in label_counts if r["sentiment"] == label)
    print(f"  {idx} -> {label:10s} ({count:,} training samples)")

MAPPING_PATH = f"{BASE}/model/label_mapping.json"
with open(MAPPING_PATH, "w") as f:
    json.dump(label_mapping, f, indent=2)

print(f"\nSaved to : {MAPPING_PATH}")
print("\nTraining complete. Ready to build the streaming pipeline.")

Label mapping (numeric prediction -> sentiment string):
  0 -> positive   (245,435 training samples)
  1 -> negative   (45,659 training samples)
  2 -> neutral    (23,806 training samples)

Saved to : /Users/beethoven/BigData/AmazonReview/model/label_mapping.json

Training complete. Ready to build the streaming pipeline.
